# 🛡️ StegShield — Full Training Pipeline on Colab T4 GPU

**Run cells top-to-bottom. This notebook trains all branches + fusion on BOSSBase.**

| Stage | Estimated Time (T4 GPU) | Target Accuracy |
|-------|------------------------|-----------------|
| Branch A (Pixel CNN) | ~45–90 min | ≥82% |
| Branch B (DCT CNN) | ~45–90 min | ≥76% |
| Branch C (Stats MLP) | ~15–30 min | ≥68% |
| Fusion (Head only) | ~30–60 min | ≥85% |
| Joint Fine-tuning | ~30–60 min | ≥88% |
| **Total** | **~3–5 hours** | — |

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU (T4)')

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# IMPORTANT: Upload the compressed EDI_Project.zip file to the root of your Google Drive.
ZIP_DRIVE_PATH = "/content/drive/MyDrive/EDI_Project.zip"
print(f"Zip path: {ZIP_DRIVE_PATH}")

In [ ]:
# ── Cell 3: Setup Project (Unzip to fast local SSD) ───────────────────────
import os, sys

# Copy zip to fast local SSD and extract (much faster than reading from Drive)
!cp "{ZIP_DRIVE_PATH}" /content/EDI_Project.zip
!unzip -q /content/EDI_Project.zip -d /content/
os.chdir("/content/EDI_Project")
sys.path.insert(0, "src")

# Install missing dependencies
!pip install -q scipy scikit-learn seaborn plotly

print("Working directory:", os.getcwd())
print("Data/clean images:", len(list(__import__("pathlib").Path("data/clean").glob("*.pgm"))))
print("Data/stego images:", len(list(__import__("pathlib").Path("data/stego").glob("*.png"))))

In [ ]:
# ── Cell 4: Quick Sanity Test (runs in ~30 seconds) ───────────────────────
# Verify the full pipeline works before committing 3 hours of GPU time
print('Running quick sanity test (20 images, 2 epochs)...')
!python src/training/train.py --mode branch_a --max_images 100 --epochs 3
print('\n✅ Sanity test passed — pipeline is working!')

In [ ]:
# ── Cell 5: STAGE 1 — Train Branch A (Pixel CNN + SRM) ───────────────────
# Detects: LSB, WOW, HILL (spatial domain hiding)
# Target:  ≥82% accuracy on BOSSBase 0.4 bpp
print('='*60)
print('STAGE 1: Training Branch A — Pixel Domain CNN')
print('='*60)
!python src/training/train.py --mode branch_a --epochs 80
print('\n✅ Branch A training complete!')
print('Checkpoint saved to: weights/branch_a_best.pt')

In [ ]:
# ── Cell 6: STAGE 2 — Train Branch B (DCT Frequency CNN) ─────────────────
# Detects: J-UNIWARD, F5, UERD (JPEG / frequency-domain hiding)
# Target:  ≥76% accuracy
print('='*60)
print('STAGE 2: Training Branch B — DCT Frequency CNN')
print('='*60)
!python src/training/train.py --mode branch_b --epochs 80
print('\n✅ Branch B training complete!')
print('Checkpoint saved to: weights/branch_b_best.pt')

In [ ]:
# ── Cell 7: STAGE 3 — Train Branch C (Statistical MLP) ───────────────────
# Detects: Chi-square anomalies, histogram equalisation, LSB statistical tests
# Target:  ≥68% accuracy
print('='*60)
print('STAGE 3: Training Branch C — Statistical MLP')
print('='*60)
!python src/training/train.py --mode branch_c --epochs 80
print('\n✅ Branch C training complete!')
print('Checkpoint saved to: weights/branch_c_best.pt')

In [ ]:
# ── Cell 7.5: STAGE 4 — Train Branch D (Generative AI CNN) ───────────────────────
# Detects: Diffusion/GAN steganography
# Target:  ≥85% accuracy
print("="*60)
print("STAGE 4: Training Branch D — Generative AI CNN")
print("="*60)
!python src/training/train.py --mode branch_d --epochs 80
print("\n✅ Branch D training complete!")
print("Checkpoint saved to: weights/branch_d_best.pt")

In [ ]:
# ── Cell 8: STAGE 4 — Train Fusion Head (branches FROZEN) ────────────────
# Only the fusion classifier head is trained here.
# Branches are FROZEN so their learned features are preserved.
# Target:  ≥80% (first pass)
print('='*60)
print('STAGE 4: Training Fusion Head (branches frozen)')
print('='*60)
!python src/training/train.py --mode fusion --epochs 60 --lr 1e-3
print('\n✅ Fusion head training complete!')
print('Checkpoint saved to: weights/fusion_best.pt')

In [ ]:
# ── Cell 9: STAGE 5 — Joint Fine-tuning (ALL params unfrozen) ────────────
# All branches + fusion trained together at very low LR.
# This is where the model reaches its peak accuracy (≥85-88%)
# CRITICAL: Use 1e-5 LR to avoid catastrophic forgetting of branch features.
print('='*60)
print('STAGE 5: Joint Fine-tuning (all params, LR=1e-5)')
print('='*60)
!python src/training/train.py --mode joint --epochs 30 --lr 1e-5
print('\n✅ Joint fine-tuning complete!')
print('Best model saved to: weights/fusion_best.pt')

In [ ]:
# ── Cell 10: Evaluation — Full Metrics Report ─────────────────────────────
print('='*60)
print('EVALUATION: Generating full metrics report')
print('='*60)
!python src/training/evaluate.py \
    --checkpoint weights/fusion_best.pt \
    --mode fusion \
    --ablation \
    --roc
print('\nResults saved to: results/')

In [ ]:
# ── Cell 11: INT8 Quantization ────────────────────────────────────────────
print('='*60)
print('QUANTIZATION: INT8 CPU-only deployment model')
print('='*60)
!python src/deploy/quantize.py \
    --model_path weights/fusion_best.pt \
    --output_path weights/fusion_int8.pt

import os
orig_mb = os.path.getsize('weights/fusion_best.pt') / 1e6
int8_mb = os.path.getsize('weights/fusion_int8.pt') / 1e6
print(f'\nModel size: {orig_mb:.1f} MB → {int8_mb:.1f} MB ({100*(1-int8_mb/orig_mb):.0f}% reduction)')

In [ ]:
# ── Cell 12: Ablation Study (for IEEE paper) ──────────────────────────────
print('='*60)
print('ABLATION STUDY: 8-way configuration test')
print('='*60)
!python ablation_study.py \
    --fusion_ckpt weights/fusion_best.pt \
    --max_images 2000
print('\nAblation table saved to: results/ablation_results.csv')

In [ ]:
# ── Cell 13: Save Everything Back to Google Drive ─────────────────────────
import shutil

print('Saving trained weights and results back to Google Drive...')
!cp -r /content/EDI_Project/weights "{PROJECT_DRIVE_PATH}/"
!cp -r /content/EDI_Project/results "{PROJECT_DRIVE_PATH}/"

print('\n✅ Saved to Google Drive!')
print(f'Weights: {PROJECT_DRIVE_PATH}/weights/')
print(f'Results: {PROJECT_DRIVE_PATH}/results/')
print('\nDownload weights/ folder to your laptop to use with the Streamlit demo.')

In [ ]:
# ── Cell 14: (OPTIONAL) Baseline Comparison — Xu-Net ─────────────────────
# Train the simplified Xu-Net baseline for the comparison table in the paper.
# This is required by the IEEE paper to prove your model outperforms baselines.
print('='*60)
print('BASELINE: Training Xu-Net for comparison table')
print('='*60)
!python src/baselines/xunet_baseline.py --train --epochs 50
print('\nXu-Net baseline trained. Results in results/xunet_evaluation.json')

## 📊 Expected Results for IEEE Paper

| Model | LSB Acc | S-UNIWARD Acc | J-UNIWARD Acc | Avg |
|-------|---------|---------------|---------------|-----|
| Xu-Net (baseline) | ~82% | ~68% | ~59% | ~70% |
| Yedroudj-Net (baseline) | ~84% | ~72% | ~63% | ~73% |
| **Our MBCSS (fusion)** | **~88%** | **~82%** | **~76%** | **~82%** |

- **Results are estimates** — your actual numbers will vary.
- **Even 85% average beats the baselines** — that's a valid IEEE paper contribution.
- Report the **ACTUAL numbers** from your training run, not these estimates.

## 📁 Output Files After Training

```
weights/
├── branch_a_best.pt     ← Branch A (pixel CNN)
├── branch_b_best.pt     ← Branch B (DCT CNN)
├── branch_c_best.pt     ← Branch C (stats MLP)
├── fusion_best.pt       ← Full fusion model (FP32)
└── fusion_int8.pt       ← Quantized CPU model

results/
├── history_branch_a.json      ← Training loss/acc curves
├── history_branch_b.json
├── history_branch_c.json
├── history_fusion.json
├── confusion_matrix.png       ← For paper Figure
├── roc_curve.png              ← For paper Figure
├── ablation_results.csv       ← Ablation study table
└── xunet_evaluation.json      ← Baseline comparison
```